---
# Week 3 - Model Quantization
--- 

Task 1: Tensor Quantization and Dequantization

File: task_01_basic_quantization.ipynb

Objective: Implement basic affine INT8 quantization and dequantization using NumPy. Scale and zero points are given so you can focus purely on the quantization and dequantization formulas.

In [5]:
import torch

In [9]:

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_printoptions(precision=4)

def quantize_tensor(tensor:torch.Tensor, scale:float, zero_point:int):
    if -128 <= zero_point <= 127:
        # divide the input by scale then add the zero point
        return torch.clamp(torch.round(tensor / scale + 1e-9) + zero_point, min=-128, max=127).to(torch.int8)
    return None

# we can use torch.quantize_per_tensor() function also (the i/p tensor
# must be a result of torch.quantize_per_tensor() or related function)

def dequantize_tensor(quantized_tensor: torch.Tensor, scale: float, zero_point: int):
    if -128 <= zero_point <= 127:
        # Recover the original floating-point value by removing the zero point and scale it back to original range
        return (quantized_tensor - zero_point) * (scale + 1e-9) 

# we can use
def calculate_error(original_tensor: torch.tensor, quantized_tensor: torch.tensor):
    
    if original_tensor.shape != quantized_tensor.shape:
        return None
    # check device compatibility
    if original_tensor.device != quantized_tensor.device:
        original_tensor.to(device=device)
        quantized_tensor.to(device=device)
    
    # Calculate Absoulte Error
    abs_err = torch.abs(original_tensor - quantized_tensor)
    means_abs_err = abs_err.mean()
    # mean_abs_err = torch.nn.L1Loss(original_tensor, quantized_tensor)
    max_abs_err = abs_err.max()
    return {"abs_err": abs_err, "mean_abs_err": means_abs_err,
            "max_abs_err": max_abs_err}

In [10]:
input1 = torch.tensor([1.0, -0.5, 3.2, -2.8, 0.0]).to(device=device)
scale, zero_point = 0.025, 0
quantized_tensor = quantize_tensor(input1, scale, zero_point)

dequantized_tensor = dequantize_tensor(quantized_tensor, scale, zero_point)
err_metrics = calculate_error(input1, dequantized_tensor)

In [11]:
print("Output: ")
print(f'''Original Tensor: {input1}\n
Quantized Tensor: {quantized_tensor}\n
Dequantized Tensor: {dequantized_tensor}\n
Absoulute error per element: {err_metrics["abs_err"]}\n
Mean Absolute error: {err_metrics["mean_abs_err"]:0.4f}\n
Max Absolute error: {err_metrics["max_abs_err"]:0.4f}''')


Output: 
Original Tensor: tensor([ 1.0000, -0.5000,  3.2000, -2.8000,  0.0000])

Quantized Tensor: tensor([  40,  -20,  127, -112,    0], dtype=torch.int8)

Dequantized Tensor: tensor([ 1.0000, -0.5000,  3.1750, -2.8000,  0.0000])

Absoulute error per element: tensor([0.0000, 0.0000, 0.0250, 0.0000, 0.0000])

Mean Absolute error: 0.0050

Max Absolute error: 0.0250
